# 考古墓葬数据库 - 数据探索

SQLite 数据库查询与可视化

**当前数据**: 898 座墓葬, 2,291 件器物 (11 个报告)

In [ ]:
import sqlite3, json
from pathlib import Path

DB = Path('../kaogu.db')
conn = sqlite3.connect(DB)
conn.row_factory = sqlite3.Row

print(f'数据库: {DB.absolute()}')
print(f'大小: {DB.stat().st_size / 1024:.1f} KB')

In [ ]:
# 各表记录数
tables = ['reports', 'sites', 'tombs', 'artifacts']
for t in tables:
    count = conn.execute(f'SELECT count(*) FROM {t}').fetchone()[0]
    print(f'{t}: {count}')

In [ ]:
# 按朝代统计
dynasties = conn.execute('''
    SELECT dynasty, COUNT(*) as count 
    FROM tombs 
    WHERE dynasty IS NOT NULL
    GROUP BY dynasty 
    ORDER BY count DESC
''').fetchall()

print('按朝代统计:')
for d in dynasties:
    print(f'  {d["dynasty"]}: {d["count"]} 座')

In [ ]:
# 按墓葬形制统计
structures = conn.execute('''
    SELECT structure, COUNT(*) as count 
    FROM tombs 
    WHERE structure IS NOT NULL
    GROUP BY structure 
    ORDER BY count DESC
    LIMIT 10
''').fetchall()

print('按墓葬形制统计:')
for s in structures:
    print(f'  {s["structure"]}: {s["count"]} 座')

In [ ]:
# 按器物材质统计
materials = conn.execute('''
    SELECT material, COUNT(*) as count 
    FROM artifacts 
    WHERE material IS NOT NULL
    GROUP BY material 
    ORDER BY count DESC
''').fetchall()

print('按器物材质统计:')
for m in materials:
    print(f'  {m["material"]}: {m["count"]} 件')

In [ ]:
# 按报告统计
reports = conn.execute('''
    SELECT r.title, 
           COUNT(DISTINCT t.id) as tomb_count,
           COUNT(a.id) as artifact_count
    FROM reports r
    LEFT JOIN tombs t ON t.report_id = r.id
    LEFT JOIN artifacts a ON a.tomb_id = t.id
    GROUP BY r.id
    ORDER BY tomb_count DESC
''').fetchall()

print('按报告统计:')
for r in reports:
    print(f'  {r["title"]}: {r["tomb_count"]} 墓, {r["artifact_count"]} 器物')

In [ ]:
# 墓葬概览（带坐标的）
tombs_with_coords = conn.execute('''
    SELECT t.tomb_number, t.dynasty, t.structure, t.length, t.width, t.depth,
           s.name as site_name, s.province, s.latitude, s.longitude
    FROM tombs t
    JOIN sites s ON t.site_id = s.id
    WHERE s.latitude IS NOT NULL
    LIMIT 20
''').fetchall()

print('墓葬概览（有坐标）:')
for t in tombs_with_coords:
    print(f'  {t["tomb_number"]}: {t["dynasty"] or "?"} | {t["site_name"]} | {t["province"]}')

In [ ]:
# 导出 GeoJSON
geojson = {'type': 'FeatureCollection', 'features': []}

rows = conn.execute('''
    SELECT s.name, s.province, s.city, s.latitude, s.longitude,
           COUNT(t.id) as tomb_count,
           GROUP_CONCAT(DISTINCT t.dynasty) as dynasties
    FROM sites s
    LEFT JOIN tombs t ON t.site_id = s.id
    WHERE s.latitude IS NOT NULL
    GROUP BY s.id
''').fetchall()

for r in rows:
    feature = {
        'type': 'Feature',
        'geometry': {'type': 'Point', 'coordinates': [r['longitude'], r['latitude']]},
        'properties': {
            'name': r['name'],
            'province': r['province'] or '',
            'city': r['city'] or '',
            'tomb_count': r['tomb_count'],
            'dynasties': r['dynasties'] or ''
        }
    }
    geojson['features'].append(feature)

output = Path('../data/sites.geojson')
output.parent.mkdir(exist_ok=True)
with open(output, 'w') as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)

print(f'导出 {len(geojson["features"])} 个站点到 {output}')

In [ ]:
conn.close()
print('Done')